# InstantID Mobile — Full Export Pipeline

Pipeline chuyển InstantID → ONNX → INT8 → OnnxStream cho mobile.

| Bước | Cell | Thời gian | Disk cần |
|------|------|-----------|----------|
| 0. Setup + tải models | 0-1 | ~15 phút | ~15 GB |
| 1. Fuse LCM-LoRA | 2 | ~5 phút | ~13 GB peak |
| 2. Export ONNX | 3a-3b | ~30-60 phút | ~20 GB peak |
| 3. Cleanup + Quantize INT8 | 4a-4b | ~20-40 phút | ~15 GB peak |
| 4. Test pipeline | 5 | ~5 phút | ~3 GB |
| 5. Save to Drive | 6 | ~5 phút | — |

**Runtime:** T4 GPU (Colab free) — A100 nhanh hơn nhưng không bắt buộc.

**Google Drive cache:** Models + kết quả được cache trên Drive. Session mới restore trong ~1 phút thay vì tải lại 30 phút.

**QUAN TRỌNG:** Chạy các cell THEO THỨ TỰ. Không skip cell.

---
## Cell 0 — Mount Drive + Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'
!mkdir -p "{DRIVE_CACHE}"/{checkpoints,sdxl-base-lcm,onnx-int8}

# Core dependencies
!pip install -q \
    "diffusers>=0.28.0" \
    "transformers>=4.40.0" \
    "huggingface_hub>=0.23.0" \
    "accelerate>=0.30.0" \
    "safetensors" \
    "peft" \
    "onnx" \
    "onnxscript" \
    "onnxruntime" \
    "onnxslim" \
    "insightface" \
    "opencv-python" \
    "Pillow" \
    "scipy"

import torch, diffusers, onnxruntime
print(f"PyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}")
print(f"diffusers {diffusers.__version__}")
print(f"onnxruntime {onnxruntime.__version__}")
!df -h /content | tail -1

---
## Cell 1 — Clone Repo + Download Models

In [ ]:
import os, shutil
os.chdir('/content')

# Clone repo
if not os.path.exists('Instantid-mobile/.git'):
    !git clone https://github.com/Hert4/Instantid-mobile
else:
    !cd Instantid-mobile && git pull origin main

os.chdir('/content/Instantid-mobile')
print(f"Working dir: {os.getcwd()}")

In [ ]:
# ── Helper: restore từ Drive cache ──────────────────────────────────────
import os
DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'

def dir_size_mb(path):
    """Tính tổng size (MB) của 1 folder."""
    if not os.path.exists(path):
        return 0
    total = 0
    for dp, _, fns in os.walk(path):
        for f in fns:
            try:
                total += os.path.getsize(os.path.join(dp, f))
            except OSError:
                pass
    return total / 1024**2

def restore_from_drive(name, min_mb=10):
    """Copy từ Drive cache về local nếu có data."""
    drive_path = f'{DRIVE_CACHE}/{name}'
    local_path = f'/content/Instantid-mobile/{name}'
    drive_mb = dir_size_mb(drive_path)
    local_mb = dir_size_mb(local_path)
    if drive_mb > min_mb and local_mb < min_mb:
        print(f'  Restoring {name} from Drive ({drive_mb:.0f} MB)...')
        !cp -r "{drive_path}" "{local_path}"
        print(f'  ✅ Restored {name}')
        return True
    elif local_mb >= min_mb:
        print(f'  ✅ {name} already exists locally ({local_mb:.0f} MB)')
        return True
    return False

def save_to_drive(name, min_mb=10):
    """Copy local → Drive cache."""
    src = f'/content/Instantid-mobile/{name}'
    dst = f'{DRIVE_CACHE}/{name}'
    src_mb = dir_size_mb(src)
    dst_mb = dir_size_mb(dst)
    if src_mb < min_mb:
        print(f'  [skip] {name} too small ({src_mb:.0f} MB) — probably incomplete')
        return
    if abs(src_mb - dst_mb) < min_mb:
        print(f'  [skip] {name} already cached on Drive ({dst_mb:.0f} MB)')
        return
    print(f'  Saving {name} to Drive ({src_mb:.0f} MB)...')
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    !cp -r "{src}" "{dst}"
    print(f'  ✅ Cached {name}')

print('Helpers loaded ✅')

In [ ]:
# ── Download checkpoints (check từng file, không skip cả thư mục) ───────
os.chdir('/content/Instantid-mobile')
os.makedirs('checkpoints/ControlNetModel', exist_ok=True)
os.makedirs('models/antelopev2', exist_ok=True)

# Restore checkpoints từ Drive nếu có
restore_from_drive('checkpoints', min_mb=100)

from huggingface_hub import hf_hub_download, snapshot_download

# ── IP-Adapter ──────────────────────────────────────────────────────────
print('=== IP-Adapter ===')
if not os.path.exists('checkpoints/ip-adapter.bin'):
    print('  Downloading ip-adapter.bin (~1.6 GB)...')
    hf_hub_download("InstantX/InstantID", "ip-adapter.bin",
                    local_dir="./checkpoints")
print(f'  ✅ ip-adapter.bin')

# ── ControlNet ──────────────────────────────────────────────────────────
print('=== ControlNet ===')
if not os.path.exists('checkpoints/ControlNetModel/config.json'):
    print('  Downloading ControlNetModel (~2.5 GB)...')
    snapshot_download("InstantX/InstantID",
                      allow_patterns=["ControlNetModel/*"],
                      local_dir="./checkpoints")
print(f'  ✅ ControlNetModel/')

# ── LCM-LoRA (từ repo KHÁC — latent-consistency, không phải InstantX) ──
print('=== LCM-LoRA ===')
if not os.path.exists('checkpoints/pytorch_lora_weights.safetensors'):
    print('  Downloading LCM-LoRA weights (~400 MB)...')
    hf_hub_download("latent-consistency/lcm-lora-sdxl",
                    "pytorch_lora_weights.safetensors",
                    local_dir="./checkpoints")
print(f'  ✅ pytorch_lora_weights.safetensors')

# Cache lên Drive
save_to_drive('checkpoints')

# ── InsightFace antelopev2 ──────────────────────────────────────────────
print('\n=== InsightFace ===')
import glob
scrfd_files = glob.glob('models/**/scrfd_10g_bnkps.onnx', recursive=True)
if not scrfd_files:
    print('  Downloading InsightFace antelopev2...')
    try:
        import insightface
        from insightface.app import FaceAnalysis
        app = FaceAnalysis(name='antelopev2', root='./models',
                           providers=['CPUExecutionProvider'])
        app.prepare(ctx_id=-1, det_size=(640, 640))
        print('  ✅ InsightFace (via insightface package)')
    except Exception as e1:
        print(f'  insightface auto-download failed: {e1}')
        try:
            for f in ['scrfd_10g_bnkps.onnx', 'glintr100.onnx']:
                hf_hub_download("DIAMONIK7777/antelopev2", f,
                                local_dir="./models/antelopev2")
            print('  ✅ InsightFace (from HF fallback)')
        except Exception as e2:
            print(f'  ⚠️ Both downloads failed. Manual: place scrfd_10g_bnkps.onnx + glintr100.onnx in models/antelopev2/')
else:
    print(f'  ✅ Already exists: {scrfd_files[0]}')

# ── Final verification ──────────────────────────────────────────────────
print('\n=== Verification ===')
files = {
    'checkpoints/ip-adapter.bin': 'IP-Adapter',
    'checkpoints/ControlNetModel/config.json': 'ControlNet',
    'checkpoints/pytorch_lora_weights.safetensors': 'LCM-LoRA',
}
all_ok = True
for p, name in files.items():
    ok = os.path.exists(p)
    print(f'  {"✅" if ok else "❌ MISSING":12s} {name:20s} {p}')
    all_ok = all_ok and ok

scrfd = glob.glob('models/**/scrfd_10g_bnkps.onnx', recursive=True)
print(f'  {"✅" if scrfd else "❌ MISSING":12s} {"InsightFace":20s} {scrfd[0] if scrfd else "NOT FOUND"}')

assert all_ok, "Some checkpoints missing! Check errors above."
!df -h /content | tail -1

---
## Cell 2 — Fuse LCM-LoRA (SDXL + LoRA → sdxl-base-lcm)

Merge LCM-LoRA vào SDXL UNet → inference chỉ cần 4-8 steps thay vì 20-50.

`--delete_source` xóa sdxl-base TRƯỚC KHI save → giải phóng ~6 GB.

In [ ]:
os.chdir('/content/Instantid-mobile')

if os.path.exists('sdxl-base-lcm/model_index.json'):
    print('✅ sdxl-base-lcm đã tồn tại — skip')
elif restore_from_drive('sdxl-base-lcm', min_mb=1000):
    assert os.path.exists('sdxl-base-lcm/model_index.json'), 'Restore failed!'
else:
    # Cần download SDXL base trước
    if not os.path.exists('sdxl-base/model_index.json'):
        print('Downloading SDXL base (safetensors only, ~6.5 GB)...')
        from huggingface_hub import snapshot_download
        snapshot_download(
            "stabilityai/stable-diffusion-xl-base-1.0",
            local_dir="./sdxl-base",
            # CHỈ tải safetensors — bỏ bin/ckpt/flax/onnx = tiết kiệm ~30 GB
            ignore_patterns=["*.bin", "*.ckpt", "flax_model*", "tf_model*",
                             "*.onnx*", "*.msgpack", "*.xml", "*.pb"],
        )
        print(f'✅ Downloaded SDXL base')
    else:
        print(f'✅ sdxl-base already exists')

    !du -sh sdxl-base
    !df -h /content | tail -1

    # Fuse
    print('\nFusing LCM-LoRA...')
    !python scripts/fuse_lcm_lora.py --device cuda --delete_source

    assert os.path.exists('sdxl-base-lcm/model_index.json'), \
        'Fuse FAILED! Check errors above.'

    # Cache kết quả
    save_to_drive('sdxl-base-lcm', min_mb=1000)

print(f'\n✅ Fuse OK! sdxl-base-lcm = {dir_size_mb("sdxl-base-lcm"):.0f} MB')

In [ ]:
# Cleanup: xóa sdxl-base gốc + HF cache
import shutil
for path in ['./sdxl-base', os.path.expanduser('~/.cache/huggingface/hub')]:
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f'Freed {path}')

!df -h /content | tail -1

---
## Cell 3a — Export ONNX (define helpers)

Export từng component riêng để tránh OOM.

Models lớn (> 1.5 GB params) tự động dùng **external data format** để bypass protobuf 2GB limit:
- `.onnx` file chứa graph structure (nhỏ)
- `weights.pb` file chứa weights (lớn)

> **`SKIP_UNET = True`** → bỏ qua export + quantize UNet để tránh OOM.  
> Pipeline vẫn dùng PyTorch UNet khi inference (chỉ các component còn lại chạy ONNX).

In [ ]:
os.chdir('/content/Instantid-mobile')
os.makedirs('onnx', exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
# ⚙️  TOGGLE: đặt True để bỏ qua UNet (tránh OOM trên T4 / máy RAM thấp)
SKIP_UNET = True   # ← đổi thành False nếu muốn export UNet
# ─────────────────────────────────────────────────────────────────────────

if SKIP_UNET:
    print('⚠️  SKIP_UNET=True — UNet sẽ KHÔNG được export/quantize.')
    print('    Pipeline inference sẽ dùng PyTorch UNet thay thế.')

# Expected TOTAL sizes per component dir (MB) — includes external data (.pb)
EXPECTED_SIZES = {
    'ip_adapter':     ('onnx/ip_adapter',     50),
    'controlnet':     ('onnx/controlnet',     2000),
    'text_encoder':   ('onnx/text_encoder',   300),
    'text_encoder_2': ('onnx/text_encoder_2', 700),
    'vae':            ('onnx/vae_decoder',    80),
    'unet':           ('onnx/unet',           4000),
}

def dir_total_mb(dirpath):
    """Total size of all files in a directory (MB)."""
    if not os.path.isdir(dirpath):
        return 0
    return sum(
        os.path.getsize(os.path.join(dirpath, f))
        for f in os.listdir(dirpath)
        if os.path.isfile(os.path.join(dirpath, f))
    ) / 1024**2

def verify_onnx(name):
    """Check ONNX dir exists and total size (incl. external data) is reasonable."""
    dirpath, min_mb = EXPECTED_SIZES[name]
    onnx_file = os.path.join(dirpath, 'model.onnx')
    if not os.path.exists(onnx_file):
        return False
    actual_mb = dir_total_mb(dirpath)
    if actual_mb < min_mb:
        print(f'  ⚠️  {name}: {actual_mb:.0f} MB < {min_mb} MB — incomplete, will re-export')
        # Clean the dir for re-export
        import shutil
        shutil.rmtree(dirpath, ignore_errors=True)
        return False
    print(f'  ✅ {name}: {actual_mb:.0f} MB')
    return True

print('=== Current ONNX files ===')
for name in EXPECTED_SIZES:
    if name == 'unet' and SKIP_UNET:
        print(f'  ⏭️  unet: SKIPPED (SKIP_UNET=True)')
        continue
    verify_onnx(name)

In [ ]:
# ── Export IP-Adapter + ControlNet (không cần load SDXL pipeline) ──────
os.chdir('/content/Instantid-mobile')
import gc, torch

if not verify_onnx('ip_adapter'):
    print('\n--- Exporting IP-Adapter ---')
    !python scripts/export_all.py --only ip_adapter
    assert verify_onnx('ip_adapter'), 'IP-Adapter export FAILED!'

if not verify_onnx('controlnet'):
    print('\n--- Exporting ControlNet ---')
    !python scripts/export_all.py --only controlnet
    assert verify_onnx('controlnet'), 'ControlNet export FAILED!'

gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
!df -h /content | tail -1

In [ ]:
# ── Export Text Encoders + VAE (+ UNet nếu SKIP_UNET=False) ─────────────
os.chdir('/content/Instantid-mobile')

assert os.path.exists('sdxl-base-lcm/model_index.json'), \
    'sdxl-base-lcm MISSING! Chạy lại Cell 2 (Fuse).'

# Danh sách component cần export (bỏ unet nếu SKIP_UNET=True)
export_plan = {
    'text_encoders': ['text_encoder', 'text_encoder_2'],
    'vae':           ['vae'],
}
if not SKIP_UNET:
    export_plan['unet'] = ['unet']
else:
    print('⏭️  Skipping UNet export (SKIP_UNET=True)\n')

for component, verify_keys in export_plan.items():
    all_ok = all(verify_onnx(k) for k in verify_keys)
    if not all_ok:
        print(f'\n--- Exporting {component} ---')
        !python scripts/export_all.py --only {component}
        for k in verify_keys:
            assert verify_onnx(k), f'{k} export FAILED! Check errors above.'

# Tổng kết
print('\n=== All ONNX exports ===')
total_mb = 0
for name, (dirpath, _) in EXPECTED_SIZES.items():
    if name == 'unet' and SKIP_UNET:
        print(f'  {"unet":20s} {"SKIPPED":>8s}')
        continue
    mb = dir_total_mb(dirpath)
    if mb > 0:
        total_mb += mb
        print(f'  {name:20s} {mb:>8.0f} MB')
print(f'  {"TOTAL":20s} {total_mb:>8.0f} MB ({total_mb/1024:.1f} GB)')
!df -h /content | tail -1

In [ ]:
# Cleanup TRƯỚC quantize: xóa sdxl-base-lcm + HF cache để free disk cho quantize
# (onnx/ vẫn giữ — cần cho quantize!)
import shutil

if os.path.exists('sdxl-base-lcm'):
    shutil.rmtree('sdxl-base-lcm', ignore_errors=True)
    print('Freed sdxl-base-lcm (~6.5 GB)')

hf_cache = os.path.expanduser('~/.cache/huggingface/hub')
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache, ignore_errors=True)
    print('Freed HF cache')

# Free GPU memory — quantize chạy CPU
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('Freed GPU memory')

!df -h /content | tail -1

---
## Cell 4b — Quantize INT8

Dynamic INT8 quantization: weights INT8 offline, activations FP32 at runtime.

~75% size reduction. VAE giữ FP16 (nhạy cảm với quantization).

Quantize từng component riêng để quản lý RAM:
- Nhỏ trước (text_encoder, ip_adapter, vae)
- Lớn sau (controlnet, text_encoder_2)
- UNet: **bị bỏ qua nếu `SKIP_UNET=True`** (tránh OOM)
- Nếu `SKIP_UNET=False`: UNet dùng `optimize_model=False` + `use_external_data_format=True`

In [ ]:
os.chdir('/content/Instantid-mobile')
import gc

# Quantize từng component — nhỏ trước, UNet cuối cùng (nếu không skip)
components = ['text_encoder', 'ip_adapter', 'vae', 'text_encoder_2', 'controlnet']
if not SKIP_UNET:
    components.append('unet')
else:
    print('⏭️  Skipping UNet quantize (SKIP_UNET=True)\n')

for comp in components:
    print(f'\n{"="*50}')
    print(f'Quantizing: {comp}')
    print(f'{"="*50}')

    # Free memory trước mỗi component
    gc.collect()

    !python scripts/quantize_all.py --skip_slim --only {comp}

    # Verify output exists
    import glob
    outs = glob.glob(f'onnx-int8/{comp}*/*.onnx') + glob.glob(f'onnx-int8/{comp}*/*.pb')
    if outs:
        total = sum(os.path.getsize(f) for f in outs) / 1024**2
        print(f'  -> {comp}: {total:.0f} MB')
    else:
        # VAE uses vae_decoder dir
        outs2 = glob.glob(f'onnx-int8/vae_decoder/*.onnx')
        if outs2:
            total = sum(os.path.getsize(f) for f in outs2) / 1024**2
            print(f'  -> vae_decoder: {total:.0f} MB')
        else:
            print(f'  ⚠️ {comp}: no output found!')

print('\n' + '='*50)
if SKIP_UNET:
    print('Quantization done! (UNet was skipped)')
else:
    print('All quantization done!')
!df -h /content | tail -1

In [ ]:
# Verify quantized files (bao gồm cả .pb external data)
print('=== Quantized ONNX files ===')
total_mb = 0
for root, _, files in os.walk('onnx-int8'):
    for f in sorted(files):
        if f.endswith(('.onnx', '.pb', '.onnx_data')):
            p = os.path.join(root, f)
            mb = os.path.getsize(p) / 1024**2
            total_mb += mb
            rel = os.path.relpath(p, 'onnx-int8')
            print(f'  {rel:50s} {mb:>8.0f} MB')
print(f'  {"TOTAL":50s} {total_mb:>8.0f} MB ({total_mb/1024:.1f} GB)')

# Chỉ xóa onnx/ nếu quantize đã tạo output hợp lệ (> 500 MB)
import shutil
if total_mb > 500 and os.path.exists('onnx'):
    shutil.rmtree('onnx', ignore_errors=True)
    print('\nFreed onnx/ (float16 originals)')
elif total_mb <= 500:
    print('\n⚠️  Quantized output quá nhỏ hoặc chưa có — GIỮ LẠI onnx/')
    print('    Chạy lại cell quantize ở trên trước!')

!df -h /content | tail -1

---
## Cell 5 — Test Pipeline

Chạy inference hoàn chỉnh với ONNX models để verify kết quả trước khi chuyển mobile.

In [ ]:
os.chdir('/content/Instantid-mobile')

# Upload face image hoặc dùng sample
import urllib.request
os.makedirs('test_images', exist_ok=True)

FACE_IMAGE = 'test_images/face.jpg'
if not os.path.exists(FACE_IMAGE):
    # Bạn có thể upload ảnh riêng:
    # from google.colab import files
    # uploaded = files.upload()
    # FACE_IMAGE = list(uploaded.keys())[0]
    print('⚠️  Chưa có face image!')
    print('Upload bằng cách uncomment code trên, hoặc:')
    print('  from google.colab import files')
    print('  uploaded = files.upload()')
else:
    print(f'Using: {FACE_IMAGE}')

In [ ]:
# Chạy test pipeline
os.chdir('/content/Instantid-mobile')

FACE_IMAGE = 'test_images/face.jpg'  # ← đổi path nếu cần

if os.path.exists(FACE_IMAGE):
    !python scripts/test_onnx_pipeline.py \
        --face_image "{FACE_IMAGE}" \
        --onnx_dir ./onnx-int8 \
        --insightface_dir ./models/antelopev2 \
        --prompt "portrait photo of a person, professional lighting, 4k, detailed" \
        --steps 4 \
        --seed 42 \
        --output ./output/test_result.png \
        --verbose

    # Hiển thị kết quả
    if os.path.exists('./output/test_result.png'):
        from IPython.display import Image, display
        display(Image('./output/test_result.png', width=512))
        print('✅ Pipeline test passed!')
    else:
        print('❌ Output image not found — check errors above')
else:
    print('⚠️  Skip test — upload face image trước (cell trên)')
    print('Uncomment code trong cell trên để upload.')

---
## Cell 6 — Convert cho OnnxStream (Mobile)

In [ ]:
os.chdir('/content/Instantid-mobile')

if os.path.exists('scripts/convert_for_onnxstream.py'):
    !python scripts/convert_for_onnxstream.py
    !du -sh onnxstream-models/ 2>/dev/null || echo 'No output generated'
else:
    print('convert_for_onnxstream.py chưa có — dùng onnx-int8/ trực tiếp')
    print('Copy onnx-int8/ sang mobile project.')

---
## Cell 7 — Save kết quả lên Google Drive

In [ ]:
os.chdir('/content/Instantid-mobile')
DRIVE_CACHE = '/content/drive/MyDrive/instantid-mobile-cache'

# Save quantized models
save_to_drive('onnx-int8', min_mb=50)

# Save onnxstream models (nếu có)
if os.path.exists('onnxstream-models'):
    save_to_drive('onnxstream-models', min_mb=50)

# Save test output
if os.path.exists('output/test_result.png'):
    os.makedirs(f'{DRIVE_CACHE}/output', exist_ok=True)
    !cp output/test_result.png "{DRIVE_CACHE}/output/"
    print('  ✅ Saved test_result.png')

print(f'\n✅ Tất cả đã lưu tại: {DRIVE_CACHE}/')
!du -sh "{DRIVE_CACHE}"/* 2>/dev/null

In [ ]:
# (Optional) Download zip về máy local
# import shutil
# src = 'onnx-int8'  # hoặc 'onnxstream-models'
# shutil.make_archive(src, 'zip', src)
# from google.colab import files
# files.download(f'{src}.zip')

---
## Troubleshooting

| Lỗi | Giải pháp |
|------|----------|
| `ONNX file quá nhỏ (< 100 MB cho UNet)` | Protobuf 2GB limit — pull code mới có external data format fix |
| `onnxslim treo / OOM` | Dùng `--skip_slim` — đã mặc định trong notebook |
| `sdxl-base-lcm MISSING` | Chạy lại Cell 2 (Fuse). Nếu đã bị xóa, cần tải lại sdxl-base |
| `RuntimeError: dtype mismatch Half/Float` | Pull code mới — fix trong `attention_processor.py` |
| `Casting model to float32 CPU... hangs` | OOM — pull code mới, không cast float32 nữa |
| `Disk full` | Xóa `~/.cache/huggingface/hub` + restart runtime |
| `Session timeout` | Mọi thứ đã cache trên Drive — chạy lại từ Cell 0, skip tự động |

### External Data Format
Models lớn (UNet, ControlNet, text_encoder_2) dùng ONNX external data format:
```
onnx/unet/
├── model.onnx     (graph structure, ~5 MB)
└── weights.pb     (weights, ~5 GB)
```
`onnxruntime` tự đọc cả 2 files khi load model.